# 05 胜率与加数

问题与建模思路参考 Allen B. Downey *Think Bayes*（中译《贝叶斯思维》）第 5 章。

**学习目标**：
- 用胜率（odds）形式书写贝叶斯更新
- 计算似然比并应用到血迹例
- 对独立随机变量做分布加法（卷积）与混合


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt

HERE = Path.cwd()
for candidate in [HERE, HERE / "notebooks" / "bayes", HERE.parent]:
    if (candidate / "thinkbayes_mini.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break

from thinkbayes_mini import MakeDice, MakeMixture, Odds, PmfAdd, Probability

%matplotlib inline


## 1. 胜率

概率 $p$ 对应胜率

$$
o = \frac{p}{1-p},\qquad p=\frac{o}{o+1}
$$

例如 $p=0.75$ ↔ 胜率 $3:1$。


In [ ]:
for p in [0.25, 0.5, 0.75, 0.9]:
    o = Odds(p)
    print(f"p={p:.2f} → odds={o:.3f} → back={Probability(o):.2f}")


## 2. 贝叶斯定理的胜率形式

$$
O(H \mid D) = O(H)\cdot \frac{P(D \mid H)}{P(D \mid \neg H)}
= O(H)\cdot LR
$$

其中 $LR$ 为似然比。先验胜率乘上似然比即得后验胜率。


In [ ]:
def update_odds(prior_odds: float, like_h: float, like_not_h: float) -> float:
    return prior_odds * (like_h / like_not_h)


# 曲奇饼：Bowl1 vs not（Bowl2），抽到香草
prior_odds = Odds(0.5)  # 1:1
post_odds = update_odds(prior_odds, like_h=0.75, like_not_h=0.5)
print(f"后验胜率 = {post_odds:.3f}, 后验概率 = {Probability(post_odds):.3f}")


## 3. 奥利弗的血迹

简化版：人群中 60% 带指纹类型 X。现场发现类型 X 的血迹。
奥利弗有类型 X。另有一名「随机路人」假设。

在「两人血迹互相独立、且恰好来自两人之一」的粗模型下，可用似然比比较假设——
更重要的是练习：**先写清 $H$ 与 $\neg H$ 下数据如何生成，再算 $LR$**。

下面用书中常见的血型频率数字做一次胜率更新演示（频率仅作示例）。


In [ ]:
# 假设：犯罪者血型频率
freq = {"O": 0.46, "A": 0.42, "B": 0.10, "AB": 0.02}

# 场景：已知嫌疑人是 O 型；现场血为 O 型
# H = 嫌疑人就是留血者；¬H = 随机路人留血
like_h = 1.0  # 若是嫌疑人，必出 O
like_not = freq["O"]  # 随机路人恰好 O
prior_p = 0.1  # 示意：先验认为「就是他」的概率
post_odds = update_odds(Odds(prior_p), like_h, like_not)
print(f"LR = {like_h / like_not:.3f}")
print(f"先验 p={prior_p:.2f} → 后验 p={Probability(post_odds):.3f}")


## 4. 加数：两个骰子的和

独立 $X,Y$ 时，和的分布是卷积：

$$
P(Z=z) = \sum_{x} P(X=x)\,P(Y=z-x)
$$


In [ ]:
d6 = MakeDice(6)
sum2 = PmfAdd(d6, d6)
print("两枚 d6 之和:")
for z, p in sorted(sum2.Items()):
    print(f"  {z:2d}: {p:.4f}")

xs, ps = zip(*sorted(sum2.Items()))
plt.figure(figsize=(6, 3.5))
plt.bar(xs, ps)
plt.xlabel("sum")
plt.ylabel("probability")
plt.title("PmfAdd(d6, d6)")
plt.tight_layout()


## 5. 最大化与混合分布

- **最大化**：若取 $n$ 次独立抽取的最大值，CDF 满足 $F_{\max}(x)=F(x)^n$。
- **混合**：先按权重抽一个组分分布，再从该分布抽样——`MakeMixture`。


In [ ]:
d6_max3 = d6.Max(3)
print("三枚 d6 取最大，均值 ≈", round(d6_max3.Mean(), 3))

# 混合：50% 用 d6，50% 用 d12
d12 = MakeDice(12)
mix = MakeMixture([(d6, 0.5), (d12, 0.5)])

plt.figure(figsize=(7, 3.5))
plt.plot(*zip(*sorted(d6.Items())), "o-", label="d6")
plt.plot(*zip(*sorted(d12.Items())), "s-", label="d12")
plt.plot(*zip(*sorted(mix.Items())), "^-", label="0.5/0.5 mixture")
plt.xlabel("face")
plt.ylabel("p")
plt.title("混合分布")
plt.legend()
plt.tight_layout()
print("混合分布均值 ≈", round(mix.Mean(), 3))


## 小结

1. 胜率形式把更新写成「乘似然比」，适合两个假设的对照。
2. 分布不是只能更新假设，还能对随机变量做加、最大、混合等推演。
3. 前五章主线：公式 → `Suite` 框架 → 参数估计 → 共轭/先验 → 胜率与分布运算。
